## Crawler para Download de Imagens

Esse notebook tem um exemplo básico de como seria realizado o download das imagens obtidas pelo notebook de crawler de anúncios.

In [25]:
import json
from pathlib import Path
from urllib.parse import urlparse

from io import BytesIO
import requests
from PIL import Image

from simple_crawler import Anuncio

In [12]:
anuncio_raw = Path("./dados/anuncios/anuncios_go-goiania.jsonl").read_text().splitlines().pop()

In [13]:
anuncio = Anuncio(**json.loads(anuncio_raw))

In [15]:
print(anuncio.model_dump_json(indent=2))

{
  "id": 29913012,
  "titulo": "Casa térrea 4 quartos 2 suítes jardim américa casa de rua com 4 quarto(s) e 5 banheiro(s) à venda, 323.83 por r$ 1.050.000 no setor jardim américa au28218",
  "ativo": true,
  "aceita_troca": true,
  "pet_friendly": false,
  "descricao": "Casa ampla e completa na parte alta do Jardim América Excelente oportunidade para quem busca conforto e qualidade de vida em uma das regiões mais tradicionais de Goiânia. Casa com 4 quartos, sendo 2 suítes com armários em madeira maciça, ideal para famílias que valorizam espaço e funcionalidade. Ambientes bem distribuídos: * Sala de estar/música independente * Sala de estar/TV integrada à sala de jantar * Lavabo * Cozinha com armários * Área de serviço Lazer completo com: * Piscina * Jardim * Espaço gourmet com churrasqueira * Sauna * Banheiro externo A casa conta ainda com garagem para 4 carros. Localização privilegiada na parte alta do Setor Jardim América, com fácil acesso a comércios, escolas e vias principais. Áre

In [10]:
img_base_path = "https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis"

In [16]:
for img in anuncio.imagens:
    img_url = f"{img_base_path}/{img}"
    print(img_url)

https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis/783920/29913012/go-goiania-jardim-america-nao-encontrado-casa-sobrado-a-venda-4-quartos-67f8456e-1.jpg
https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis/783920/29913012/go-goiania-jardim-america-nao-encontrado-casa-sobrado-a-venda-4-quartos-67f8456e-2.jpg
https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis/783920/29913012/go-goiania-jardim-america-nao-encontrado-casa-sobrado-a-venda-4-quartos-67f8456e-3.jpg
https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis/783920/29913012/go-goiania-jardim-america-nao-encontrado-casa-sobrado-a-venda-4-quartos-67f8456e-4.jpg
https://www.chavesnamao.com.br/imn/0000x0000/N/75/imoveis/783920/29913012/go-goiania-jardim-america-nao-encontrado-casa-sobrado-a-venda-4-quartos-67f8456e-5.jpg


In [26]:
def download_jpg(url: str, filename: str, output_dir: str = ".") -> Path:
    """
    Baixa uma imagem (idealmente JPG) de `url` e salva em `output_dir`.
    Retorna o caminho absoluto do arquivo salvo.
    """
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    base = Path(filename).stem

    # Tenta forçar JPEG no servidor
    headers = {
        "Accept": "image/jpeg,image/*;q=0.8,*/*;q=0.5",
        "User-Agent": "python-requests",
        "Referer": f"{urlparse(url).scheme}://{urlparse(url).hostname}/",
    }

    resp = requests.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    data = resp.content
    ctype = (resp.headers.get("Content-Type") or "").lower()

    # Detecta formato real com Pillow
    fmt = None
    try:
        with Image.open(BytesIO(data)) as im:
            fmt = im.format  # ex.: 'JPEG', 'WEBP', 'PNG'
    except Exception:
        pass

    # Mapeia extensão correta
    ext_map = {"JPEG": ".jpg", "JPG": ".jpg", "WEBP": ".webp", "PNG": ".png", "AVIF": ".avif"}
    if fmt in ext_map:
        real_ext = ext_map[fmt]
    elif "image/jpeg" in ctype:
        real_ext = ".jpg"
    elif "image/webp" in ctype:
        real_ext = ".webp"
    elif "image/png" in ctype:
        real_ext = ".png"
    else:
        real_ext = ".jpg"  # fallback

    # Se não for JPEG e você quer padronizar, converte
    if fmt and fmt != "JPEG":
        with Image.open(BytesIO(data)) as im:
            im = im.convert("RGB")
            out_path = out_dir / f"{base}.jpg"
            im.save(out_path, format="JPEG", quality=90, optimize=True)
    else:
        out_path = out_dir / f"{base}{real_ext}"
        out_path.write_bytes(data)

    print(f"Salvo em: {out_path.resolve()} | Content-Type: {ctype!r} | Detectado: {fmt}")
    return out_path

In [29]:
diretorio = f"dados/anuncios/imagens_{anuncio.id}"
Path(diretorio).mkdir(parents=True, exist_ok=True)
print(diretorio)

dados/anuncios/imagens_29913012


In [30]:
for idx, img in enumerate(anuncio.imagens):
    img_url = f"{img_base_path}/{img}"
    img_filename = f"{idx+1}.jpg"
    download_jpg(img_url, img_filename, output_dir=diretorio)

Salvo em: /media/wd1/trabalho/extras/aula-pdm/codigos/aula-pdm-pubsub/dados/anuncios/imagens_29913012/1.jpg | Content-Type: 'image/webp' | Detectado: WEBP
Salvo em: /media/wd1/trabalho/extras/aula-pdm/codigos/aula-pdm-pubsub/dados/anuncios/imagens_29913012/2.jpg | Content-Type: 'image/webp' | Detectado: WEBP
Salvo em: /media/wd1/trabalho/extras/aula-pdm/codigos/aula-pdm-pubsub/dados/anuncios/imagens_29913012/3.jpg | Content-Type: 'image/webp' | Detectado: WEBP
Salvo em: /media/wd1/trabalho/extras/aula-pdm/codigos/aula-pdm-pubsub/dados/anuncios/imagens_29913012/4.jpg | Content-Type: 'image/webp' | Detectado: WEBP
Salvo em: /media/wd1/trabalho/extras/aula-pdm/codigos/aula-pdm-pubsub/dados/anuncios/imagens_29913012/5.jpg | Content-Type: 'image/webp' | Detectado: WEBP
